# 02 - Data Cleaning

## Missing Value Analysis

**Objective:** Investigate the amount, pattern, and continuity of missing values before deciding how they should be handled.

In [213]:
# Import pandas for data analysis
import pandas as pd

In [214]:
# Locate the raw dataset
from pathlib import Path

possible_paths = [
    Path("data/raw/household_power_consumption.txt"),
    Path("../data/raw/household_power_consumption.txt")
]

file_path = next((path for path in possible_paths if path.exists()), None)

if file_path is None:
    raise FileNotFoundError("Raw dataset not found.")

print("Dataset path:", file_path)

Dataset path: ..\data\raw\household_power_consumption.txt


In [215]:
# Load raw data and convert '?' to missing values
df = pd.read_csv(
    file_path,
    sep=";",
    na_values="?",
    low_memory=False
)

print("Dataset loaded:", df.shape)

Dataset loaded: (2075259, 9)


In [216]:
# Define electrical measurement columns
measurement_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

In [217]:
# Count missing values in each column
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_summary

,missing_count,missing_percentage
Date,0,0.00
Time,0,0.00
Global_active_power,25979,1.25
Global_reactive_power,25979,1.25
Voltage,25979,1.25
Global_intensity,25979,1.25
Sub_metering_1,25979,1.25
Sub_metering_2,25979,1.25
Sub_metering_3,25979,1.25


In [218]:
# Count rows where all measurements are missing
all_missing_mask = df[measurement_cols].isna().all(axis=1)

all_missing_count = all_missing_mask.sum()

print("Rows with all measurements missing:", all_missing_count)

Rows with all measurements missing: 25979


In [219]:
# Count rows with at least one missing measurement
any_missing_mask = df[measurement_cols].isna().any(axis=1)

any_missing_count = any_missing_mask.sum()

print("Rows with any measurement missing:", any_missing_count)

Rows with any measurement missing: 25979


In [220]:
# Count rows with only partial measurement missingness
partial_missing_count = (
    any_missing_mask & ~all_missing_mask
).sum()

print("Partially missing measurement rows:", partial_missing_count)

Partially missing measurement rows: 0


In [221]:
# Display first missing measurement records
df.loc[all_missing_mask].head(10)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6839,21/12/2006,11:23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6840,21/12/2006,11:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19724,30/12/2006,10:08:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19725,30/12/2006,10:09:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41832,14/1/2007,18:36:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61909,28/1/2007,17:13:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98254,22/2/2007,22:58:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98255,22/2/2007,22:59:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142588,25/3/2007,17:52:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190497,28/4/2007,00:21:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [222]:
# Calculate percentage of rows with missing measurements
missing_row_percentage = (
    all_missing_count / len(df) * 100
)

print(f"Missing measurement rows: {missing_row_percentage:.2f}%")

Missing measurement rows: 1.25%


In [223]:
# Create groups for consecutive missing/non-missing rows
group_id = all_missing_mask.ne(
    all_missing_mask.shift()
).cumsum()

In [224]:
# Calculate length of each consecutive missing run
missing_runs = (
    all_missing_mask[all_missing_mask]
    .groupby(group_id[all_missing_mask])
    .size()
)

print("Number of missing runs:", len(missing_runs))

Number of missing runs: 71


In [225]:
# Summarize consecutive missing run lengths
missing_runs.describe()

count      71.000000
mean      365.901408
std      1251.468043
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max      7226.000000
dtype: float64

In [226]:
# Display the ten longest missing runs
missing_runs.sort_values(
    ascending=False
).head(10)

138    7226
140    5237
14     3723
100    3305
118    3129
126    2027
104     891
32       83
78       70
34       47
dtype: int64

In [227]:
# Build a table describing each missing period
missing_run_table = (
    df.loc[all_missing_mask, ["Date", "Time"]]
    .assign(run_id=group_id[all_missing_mask].values)
    .groupby("run_id")
    .agg(
        start_date=("Date", "first"),
        start_time=("Time", "first"),
        end_date=("Date", "last"),
        end_time=("Time", "last"),
        missing_rows=("Date", "size")
    )
)

missing_run_table.head()

,start_date,start_time,end_date,end_time,missing_rows
run_id,,,,,
2,21/12/2006,11:23:00,21/12/2006,11:24:00,2
4,30/12/2006,10:08:00,30/12/2006,10:09:00,2
6,14/1/2007,18:36:00,14/1/2007,18:36:00,1
8,28/1/2007,17:13:00,28/1/2007,17:13:00,1
10,22/2/2007,22:58:00,22/2/2007,22:59:00,2


In [228]:
# Display the largest missing periods
missing_run_table.sort_values(
    "missing_rows",
    ascending=False
).head(10)

,start_date,start_time,end_date,end_time,missing_rows
run_id,,,,,
138,17/8/2010,21:02:00,22/8/2010,21:27:00,7226
140,25/9/2010,03:56:00,28/9/2010,19:12:00,5237
14,28/4/2007,00:21:00,30/4/2007,14:23:00,3723
100,13/6/2009,00:30:00,15/6/2009,07:34:00,3305
118,12/1/2010,14:53:00,14/1/2010,19:01:00,3129
126,20/3/2010,03:52:00,21/3/2010,13:38:00,2027
104,13/8/2009,05:00:00,13/8/2009,19:50:00,891
32,15/7/2007,16:49:00,15/7/2007,18:11:00,83
78,10/12/2008,10:48:00,10/12/2008,11:57:00,70


In [229]:
# Compare valid and missing measurement records
valid_rows = (~all_missing_mask).sum()

print("Total rows:", len(df))
print("Valid measurement rows:", valid_rows)
print("Missing measurement rows:", all_missing_count)

Total rows: 2075259
Valid measurement rows: 2049280
Missing measurement rows: 25979


## Missing Value Findings

- `Date` and `Time` contain no missing values.
- Each of the seven electrical measurement columns contains 25,979 missing values.
- Missing measurements represent approximately 1.25% of the dataset.
- Missingness occurs across complete measurement rows rather than isolated individual measurement columns.
- Consecutive missing periods were investigated to determine whether gaps are short or long.
- Missing values have not yet been filled or removed.

### Cleaning Consideration

Short and long missing periods should not automatically receive the same treatment.

Small gaps may potentially be interpolated, while long gaps require more careful handling to avoid creating unrealistic synthetic energy-consumption patterns.

## Missing Value Handling

A conservative missing-value strategy is applied:

- Short missing periods of 5 minutes or less are interpolated.
- Longer missing periods are retained as missing values.
- Timestamp rows are preserved to maintain the original time-series structure.
- The raw dataset is not modified.

In [230]:
# Define the maximum gap allowed for interpolation
SHORT_GAP_LIMIT = 5

print("Short-gap limit:", SHORT_GAP_LIMIT, "minutes")

Short-gap limit: 5 minutes


In [231]:
# Separate short and long missing periods
short_run_ids = missing_runs[
    missing_runs <= SHORT_GAP_LIMIT
].index

long_run_ids = missing_runs[
    missing_runs > SHORT_GAP_LIMIT
].index

print("Short missing runs:", len(short_run_ids))
print("Long missing runs:", len(long_run_ids))

Short missing runs: 55
Long missing runs: 16


In [232]:
# Mark rows belonging to short and long gaps
short_gap_mask = (
    all_missing_mask &
    group_id.isin(short_run_ids)
)

long_gap_mask = (
    all_missing_mask &
    group_id.isin(long_run_ids)
)

print("Rows in short gaps:", short_gap_mask.sum())
print("Rows in long gaps:", long_gap_mask.sum())

Rows in short gaps: 76
Rows in long gaps: 25903


In [233]:
# Create a copy for cleaning
df_clean = df.copy()

print("Cleaning copy created:", df_clean.shape)

Cleaning copy created: (2075259, 9)


In [234]:
# Generate interpolation only between valid observations
interpolated_values = (
    df_clean[measurement_cols]
    .interpolate(
        method="linear",
        limit_area="inside"
    )
)

In [235]:
# Fill only missing rows belonging to short gaps
df_clean.loc[
    short_gap_mask,
    measurement_cols
] = interpolated_values.loc[
    short_gap_mask,
    measurement_cols
]

In [236]:
# Check missing values after short-gap interpolation
remaining_missing = df_clean.isna().sum()

remaining_missing

Date                         0
Time                         0
Global_active_power      25903
Global_reactive_power    25903
Voltage                  25903
Global_intensity         25903
Sub_metering_1           25903
Sub_metering_2           25903
Sub_metering_3           25903
dtype: int64

In [237]:
# Count rows successfully interpolated
interpolated_row_count = (
    df.loc[short_gap_mask, measurement_cols]
    .isna()
    .all(axis=1)
    &
    df_clean.loc[short_gap_mask, measurement_cols]
    .notna()
    .all(axis=1)
).sum()

print("Rows interpolated:", interpolated_row_count)

Rows interpolated: 76


In [238]:
# Confirm long missing gaps remain untouched
long_gap_remaining = (
    df_clean.loc[long_gap_mask, measurement_cols]
    .isna()
    .all(axis=1)
    .sum()
)

print("Long-gap rows still missing:", long_gap_remaining)
print("Expected long-gap rows:", long_gap_mask.sum())

Long-gap rows still missing: 25903
Expected long-gap rows: 25903


In [239]:
# Compare missing values before and after cleaning
missing_comparison = pd.DataFrame({
    "before": df[measurement_cols].isna().sum(),
    "after": df_clean[measurement_cols].isna().sum()
})

missing_comparison["handled"] = (
    missing_comparison["before"] -
    missing_comparison["after"]
)

missing_comparison

,before,after,handled
Global_active_power,25979,25903,76
Global_reactive_power,25979,25903,76
Voltage,25979,25903,76
Global_intensity,25979,25903,76
Sub_metering_1,25979,25903,76
Sub_metering_2,25979,25903,76
Sub_metering_3,25979,25903,76


In [240]:
# Display a few interpolated records
df_clean.loc[
    short_gap_mask,
    ["Date", "Time"] + measurement_cols
].head(10)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6839,21/12/2006,11:23:00,0.244667,0.000000,242.106667,1.000000,0.000000,0.0,0.000000
6840,21/12/2006,11:24:00,0.245333,0.000000,241.923333,1.000000,0.000000,0.0,0.000000
19724,30/12/2006,10:08:00,6.417333,0.402000,237.230000,27.066667,37.666667,22.0,17.666667
19725,30/12/2006,10:09:00,6.616667,0.390000,237.060000,27.933333,37.333333,25.0,17.333333
41832,14/1/2007,18:36:00,3.213000,0.257000,233.000000,13.800000,0.000000,0.0,16.500000
61909,28/1/2007,17:13:00,2.221000,0.276000,235.600000,9.400000,0.000000,0.5,16.000000
98254,22/2/2007,22:58:00,3.662667,0.083333,236.210000,15.400000,0.000000,12.0,17.000000
98255,22/2/2007,22:59:00,4.279333,0.106667,236.200000,18.000000,0.000000,24.0,17.000000
142588,25/3/2007,17:52:00,0.529000,0.228000,242.595000,2.300000,0.000000,0.0,0.000000
240590,1/6/2007,19:14:00,0.452000,0.163000,234.810000,2.100000,0.000000,0.0,0.000000


In [241]:
# Compare original and cleaned values for short gaps
comparison = pd.concat(
    [
        df.loc[short_gap_mask, measurement_cols]
          .head(5)
          .add_prefix("original_"),

        df_clean.loc[short_gap_mask, measurement_cols]
          .head(5)
          .add_prefix("cleaned_")
    ],
    axis=1
)

comparison

,original_Global_active_power,original_Global_reactive_power,original_Voltage,original_Global_intensity,original_Sub_metering_1,original_Sub_metering_2,original_Sub_metering_3,cleaned_Global_active_power,cleaned_Global_reactive_power,cleaned_Voltage,cleaned_Global_intensity,cleaned_Sub_metering_1,cleaned_Sub_metering_2,cleaned_Sub_metering_3
6839,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.244667,0.000,242.106667,1.000000,0.000000,0.0,0.000000
6840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.245333,0.000,241.923333,1.000000,0.000000,0.0,0.000000
19724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.417333,0.402,237.230000,27.066667,37.666667,22.0,17.666667
19725,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.616667,0.390,237.060000,27.933333,37.333333,25.0,17.333333
41832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.213000,0.257,233.000000,13.800000,0.000000,0.0,16.500000


### Missing Value Treatment Summary

- Missing measurement records were separated into short and long consecutive gaps.
- Gaps of 5 minutes or less were treated using linear interpolation.
- Interpolation was restricted to values located between valid observations.
- Long missing periods were intentionally retained rather than generating large amounts of synthetic electricity data.
- No timestamp records were deleted.
- The original raw dataset remains unchanged.
- Further handling of long gaps can be considered during downstream preprocessing or model-specific preparation.

## Duplicate Analysis and Handling

**Objective:** Identify exact duplicate records and repeated timestamp records before deciding whether any rows should be removed.

In [242]:
# Count fully identical rows
exact_duplicates = df_clean.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Exact duplicate rows: 0


In [243]:
# Count repeated Date-Time combinations
timestamp_duplicates = df_clean.duplicated(
    subset=["Date", "Time"]
).sum()

print("Duplicate timestamps:", timestamp_duplicates)

Duplicate timestamps: 0


In [244]:
# Display exact duplicate rows
duplicate_rows = df_clean[
    df_clean.duplicated(keep=False)
]

duplicate_rows.head(10)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3


In [245]:
# Display rows sharing the same timestamp
duplicate_timestamp_rows = df_clean[
    df_clean.duplicated(
        subset=["Date", "Time"],
        keep=False
    )
]

duplicate_timestamp_rows.head(10)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3


In [246]:
# Remove exact duplicate records only
rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

rows_after = len(df_clean)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Duplicates removed:", rows_before - rows_after)

Rows before: 2075259
Rows after: 2075259
Duplicates removed: 0


In [247]:
# Confirm no exact duplicates remain
remaining_duplicates = df_clean.duplicated().sum()

print("Remaining exact duplicates:", remaining_duplicates)

Remaining exact duplicates: 0


In [248]:
# Recheck repeated timestamps after duplicate handling
remaining_timestamp_duplicates = df_clean.duplicated(
    subset=["Date", "Time"]
).sum()

print(
    "Remaining duplicate timestamps:",
    remaining_timestamp_duplicates
)

Remaining duplicate timestamps: 0


### Duplicate Analysis Findings

- Exact duplicate records were checked.
- Duplicate `Date` and `Time` combinations were also checked separately.
- No exact duplicate records were found.
- No repeated timestamp records were found.
- Therefore, no rows required removal during duplicate handling.
- The dataset row count remained unchanged after this step.

## Invalid and Inconsistent Value Checking

**Objective:** Check the electrical measurement columns for non-numeric values, infinite values, negative values, and other obvious inconsistencies before outlier analysis.

In [249]:
# Check current column data types
df_clean.dtypes

Date                         str
Time                         str
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

In [250]:
# Check whether measurement columns contain numeric data
non_numeric_summary = {}

for column in measurement_cols:
    converted = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

    non_numeric_count = (
        converted.isna().sum()
        - df_clean[column].isna().sum()
    )

    non_numeric_summary[column] = non_numeric_count

pd.Series(
    non_numeric_summary,
    name="non_numeric_values"
)

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
Name: non_numeric_values, dtype: int64

In [251]:
# Import NumPy for numeric validation
import numpy as np

In [252]:
# Count positive and negative infinity values
infinite_summary = pd.DataFrame({
    "positive_infinity": [
        np.isposinf(df_clean[col]).sum()
        for col in measurement_cols
    ],
    "negative_infinity": [
        np.isneginf(df_clean[col]).sum()
        for col in measurement_cols
    ]
}, index=measurement_cols)

infinite_summary

,positive_infinity,negative_infinity
Global_active_power,0,0
Global_reactive_power,0,0
Voltage,0,0
Global_intensity,0,0
Sub_metering_1,0,0
Sub_metering_2,0,0
Sub_metering_3,0,0


In [253]:
# Count negative values in each measurement column
negative_summary = pd.Series(
    {
        column: (df_clean[column] < 0).sum()
        for column in measurement_cols
    },
    name="negative_values"
)

negative_summary

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
Name: negative_values, dtype: int64

In [254]:
# Display rows containing any negative measurement
negative_mask = (
    df_clean[measurement_cols] < 0
).any(axis=1)

df_clean.loc[
    negative_mask,
    ["Date", "Time"] + measurement_cols
].head(20)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3


In [255]:
# Count zero values without treating them as invalid
zero_summary = pd.Series(
    {
        column: (df_clean[column] == 0).sum()
        for column in measurement_cols
    },
    name="zero_values"
)

zero_summary

Global_active_power            0
Global_reactive_power     481569
Voltage                        0
Global_intensity               0
Sub_metering_1           1880238
Sub_metering_2           1436874
Sub_metering_3            852117
Name: zero_values, dtype: int64

In [256]:
# Review minimum and maximum values
range_summary = pd.DataFrame({
    "minimum": df_clean[measurement_cols].min(),
    "maximum": df_clean[measurement_cols].max()
})

range_summary

,minimum,maximum
Global_active_power,0.076,11.122
Global_reactive_power,0.000,1.390
Voltage,223.200,254.150
Global_intensity,0.200,48.400
Sub_metering_1,0.000,88.000
Sub_metering_2,0.000,80.000
Sub_metering_3,0.000,31.000


In [257]:
# Count non-finite values in measurement columns
non_finite_summary = pd.Series(
    {
        column: (
            ~np.isfinite(
                df_clean[column].dropna()
            )
        ).sum()
        for column in measurement_cols
    },
    name="non_finite_values"
)

non_finite_summary

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
Name: non_finite_values, dtype: int64

In [258]:
# Combine invalid-value checks into one table
invalid_value_summary = pd.DataFrame({
    "non_numeric": pd.Series(non_numeric_summary),
    "negative": negative_summary,
    "non_finite": non_finite_summary,
    "zero_values": zero_summary,
    "minimum": df_clean[measurement_cols].min(),
    "maximum": df_clean[measurement_cols].max()
})

invalid_value_summary

,non_numeric,negative,non_finite,zero_values,minimum,maximum
Global_active_power,0,0,0,0,0.076,11.122
Global_reactive_power,0,0,0,481569,0.000,1.390
Voltage,0,0,0,0,223.200,254.150
Global_intensity,0,0,0,0,0.200,48.400
Sub_metering_1,0,0,0,1880238,0.000,88.000
Sub_metering_2,0,0,0,1436874,0.000,80.000
Sub_metering_3,0,0,0,852117,0.000,31.000


In [259]:
# Identify rows containing clear numeric invalidities
clear_invalid_mask = pd.Series(
    False,
    index=df_clean.index
)

for column in measurement_cols:
    clear_invalid_mask |= np.isinf(
        df_clean[column].fillna(0)
    )

print(
    "Rows with clear invalid numeric values:",
    clear_invalid_mask.sum()
)

Rows with clear invalid numeric values: 0


### Invalid and Inconsistent Value Findings

- All electrical measurement columns were checked for unexpected non-numeric values.
- No positive or negative infinite values were found.
- Negative measurement values were checked separately.
- Zero values were retained because zero consumption or zero sub-metering can represent valid observations.
- Minimum and maximum values were reviewed, but extreme values were not removed at this stage.
- Potential extreme observations will be investigated separately during outlier analysis.
- No arbitrary domain thresholds were applied without supporting evidence.